# Dataset linting

Run quality checks on a dataset with the dataset linter API. List available rules, execute a lint run, inspect results (summary, per-rule stats, issues), and browse past runs for the same dataset.

In [1]:
%pip install lightningrod-ai python-dotenv

from IPython.display import clear_output
clear_output()

## Set up the client

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/sign-up?redirect=/api) to get your API key and **$50 of free credits**.

- **Google Colab**: Go to the Secrets section (key icon in left sidebar) and add a secret named `LIGHTNINGROD_API_KEY`
- **Local Jupyter**: Set the `LIGHTNINGROD_API_KEY` environment variable, or you'll be prompted to enter it

In [2]:
from dotenv import load_dotenv
from lightningrod import LightningRod
from lightningrod.utils import config

load_dotenv()
api_key = config.get_config_value("LIGHTNINGROD_API_KEY")

lr = LightningRod(api_key=api_key)

## Choose a dataset

Set the `DATASET_ID` environment variable to lint an existing dataset. If unset, the notebook creates an empty dataset so the example still runs (replace with a dataset that has samples for meaningful lint output).

In [3]:
import os

dataset_id = os.environ.get("LIGHTNINGROD_DATASET_ID")
print("Using dataset:", dataset_id)

Using dataset: 7c88a6dd-156f-4776-992e-811af2776c8b


## List available linter rules

Each rule has a `name` and `default_severity`.

In [4]:
from pprint import pprint

rules_response = lr.datasets.linter.list_rules()
pprint([r.to_dict() for r in rules_response.rules])

[{'default_severity': 'warning', 'name': 'base_rate'},
 {'default_severity': 'warning', 'name': 'prediction_date_distribution'},
 {'default_severity': 'warning', 'name': 'forecast_horizon_distribution'},
 {'default_severity': 'warning', 'name': 'reward_distribution'},
 {'default_severity': 'warning', 'name': 'llm_contamination'},
 {'default_severity': 'warning', 'name': 'llm_context_relevance'},
 {'default_severity': 'warning', 'name': 'llm_question_comprehension'},
 {'default_severity': 'warning', 'name': 'llm_rollout_quality'}]


## Run the linter

Call `run()` with no extra arguments to use the server defaults (all rules, default sample size for LLM-backed rules). Pass `rules=[...]` and/or `sample_size=...` to override.

In [ ]:
run_result = lr.datasets.linter.run(dataset_id)

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Dataset Linter: COMPLETED                                                                                   │
│                                                                                                                 │
│    Run ID:      0050c335-f34f-4e9c-bbda-f8b33b378e7b                                                            │
│    Dataset:     7c88a6dd-156f-4776-992e-811af2776c8b                                                            │
│    Created:     2026-04-27T16:34:40.623000+00:00                                                                │
│    Updated:     2026-04-27T16:38:48.506000+00:00                                                                │
│    Sample size: 200                                                                                             │
│                                                                                                                 │
│    Issues:      61                                                                                              │
│    Severity:    error: 2, warning: 58, info: 1                                                                  │
│                                                                                                                 │
│  ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┓  │
│  ┃ Rule                                               ┃       Issues ┃ Severity             ┃       Duration ┃  │
│  ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━┩  │
│  │ llm_context_relevance                              │           53 │ warning: 53          │          80.9s │  │
│  │ llm_contamination                                  │            5 │ warning: 5           │          74.2s │  │
│  │ forecast_horizon_distribution                      │            1 │ error: 1             │           11ms │  │
│  │ prediction_date_distribution                       │            1 │ error: 1             │           21ms │  │
│  │ reward_distribution                                │            1 │ info: 1              │            8ms │  │
│  │ base_rate                                          │            0 │ 0                    │           61ms │  │
│  │ llm_question_comprehension                         │            0 │ 0                    │          80.2s │  │
│  │ llm_rollout_quality                                │            0 │ 0                    │           65ms │  │
│  └────────────────────────────────────────────────────┴──────────────┴──────────────────────┴────────────────┘  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [8]:
from lightningrod import display_lint_detailed

display_lint_detailed(run_result)

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Dataset Linter Details                                                                                      │
│                                                                                                                 │
│    Run ID:  0050c335-f34f-4e9c-bbda-f8b33b378e7b                                                                │
│    Dataset: 7c88a6dd-156f-4776-992e-811af2776c8b                                                                │
│                                                                                                                 │
│  forecast_horizon_distribution (11ms, 1 issue)                                                                  │
│   cv                                       1.014492738051946                                                    │
│   max_days                                 458.0                                                                │
│   mean_days                                100.39945652173913                                                   │
│   min_days                                 1.0                                                                  │
│   p50_days                                 67.0                                                                 │
│   p95_days                                 356.19999999999936                                                   │
│   std_days                                 101.85451954566643                                                   │
│   valid_rows                               375                                                                  │
│                                                                                                                 │
│    Issue 1: error                                                                                               │
│      Message: 7 row(s) have non-positive forecast horizon.                                                      │
│      Affected samples: 5036a3d9-85be-4c37-ac9b-115236ebc16b                                                     │
│  ac0d36c7-8bcd-43e6-a330-cd8e5c93b78e                                                                           │
│  7df02dab-7067-41ae-a6dc-8d96bf9dd192                                                                           │
│  8004f8bc-ee18-4d02-89a0-8d39fad050e2                                                                           │
│  27562434-68b9-4bb8-bdb5-f3bc5406e523                                                                           │
│  885132ee-f2d0-4baf-916d-82e69da2a3e5                                                                           │
│  88dff541-3438-4d6c-9517-3c1d85fa5944                                                                           │
│      Meta: {"count": 7}                                                                                         │
│      Tip: The resolution must occur strictly after the prediction date.                                         │
│                                                                                                                 │
│  prediction_date_distribution (21ms, 1 issue)                                                                   │
│   clump_share                         0.176                                                                     │
│   max_date                            2025-12-27T00:00:00+00:00                                                 │
│   min_date                            2024-07-30T00:00:00+00:00                                                 │
│   span_days                           515                                                                       │
│   unique_days                         45              

## Optional: run a subset of rules

Uncomment and set rule names from the list above.

In [9]:
from lightningrod import display_lint_detailed

rule_names = [r.name for r in rules_response.rules[:2]]
subset_run = lr.datasets.linter.run(dataset_id, rules=rule_names, random_sample_size=50)
display_lint_detailed(subset_run)

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Dataset Linter: COMPLETED                                                                                   │
│                                                                                                                 │
│    Run ID:      95b5574a-c1ad-4cab-a8b5-1a2b92a07b89                                                            │
│    Dataset:     7c88a6dd-156f-4776-992e-811af2776c8b                                                            │
│    Created:     2026-04-27T16:43:16.290000+00:00                                                                │
│    Updated:     2026-04-27T16:43:39.292000+00:00                                                                │
│    Sample size: 50                                                                                              │
│                                                                                                                 │
│    Issues:      1                                                                                               │
│    Severity:    error: 1                                                                                        │
│                                                                                                                 │
│  ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓  │
│  ┃ Rule                                                 ┃        Issues ┃ Severity         ┃        Duration ┃  │
│  ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩  │
│  │ prediction_date_distribution                         │             1 │ error: 1         │            14ms │  │
│  │ base_rate                                            │             0 │ 0                │            18ms │  │
│  └──────────────────────────────────────────────────────┴───────────────┴──────────────────┴─────────────────┘  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Dataset Linter Details                                                                                      │
│                                                                                                                 │
│    Run ID:  95b5574a-c1ad-4cab-a8b5-1a2b92a07b89                                                                │
│    Dataset: 7c88a6dd-156f-4776-992e-811af2776c8b                                                                │
│                                                                                                                 │
│  prediction_date_distribution (14ms, 1 issue)                                                                   │
│   clump_share                         0.176                                                                     │
│   max_date                            2025-12-27T00:00:00+00:00                                                 │
│   min_date                            2024-07-30T00:00:00+00:00                                                 │
│   span_days                           515                                                                       │
│   unique_days                         45                                                                        │
│   valid_rows                          375                                                                       │
│                                                                                                                 │
│    Issue 1: error                                                                                               │
│      Message: 7 row(s) have prediction_date >= resolution_date.                                                 │
│      Affected samples: 5036a3d9-85be-4c37-ac9b-115236ebc16b                                                     │
│  ac0d36c7-8bcd-43e6-a330-cd8e5c93b78e                                                                           │
│  7df02dab-7067-41ae-a6dc-8d96bf9dd192                                                                           │
│  8004f8bc-ee18-4d02-89a0-8d39fad050e2                                                                           │
│  27562434-68b9-4bb8-bdb5-f3bc5406e523                                                                           │
│  885132ee-f2d0-4baf-916d-82e69da2a3e5                                                                           │
│  88dff541-3438-4d6c-9517-3c1d85fa5944                                                                           │
│      Meta: {"count": 7}                                                                                         │
│      Tip: Temporal leakage: the model would see information from after it needs to predict.                     │
│                                                                                                                 │
│  base_rate (18ms, 0 issues)                                                                                     │
│   class_counts                           {"0.0": 235, "1.0": 139}                                               │
│   max_share                              0.6283422459893048                                                     │
│   num_classes                            2                                                                      │
│   total                                  374                                                                    │
│    No issues.                                                                                                   │
│                                                                                                                 │
╰───────────────────────────────────────────────────────

## Fetch a run by id and list past runs

`get_run` returns the same shape as `run`. `list_runs` returns a lightweight list of runs for the dataset.

In [ ]:
by_id = lr.datasets.linter.get_run(run_result.id)
assert by_id.id == run_result.id

In [ ]:
past = lr.datasets.linter.list_runs(dataset_id, limit=10)
pprint([r.to_dict() for r in past.runs])